## Setup for test, starting container

In [54]:
import os
import mlflow
import requests
import json
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
mlflow.set_tracking_uri("http://localhost:5100")

MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [28]:
!docker build ../04_deploy/aggregator -t ghcr.io/alelevente/aggregator:latest --no-cache
!docker run --name mlflow_auth_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/alelevente/aggregator:latest

[+] Building 0.0s (1/1)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
[+] Building 0.2s (7/9)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 675B                                       0.0s
 => WARN: SecretsUsedInArgOrEnv: Do not use ARG or ENV instructions for s  0.0s
 => WARN: SecretsUsedInArgOrEnv: Do not use ARG or ENV instructions for s  0.0s
 => WARN: SecretsUsedInArgOrEnv: Do not use ARG or ENV instructions for s  0.0s
 => [internal] load metadata for ghcr.io/alelevente/base_slim:latest       0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 64B                                           0.0s
 => [1/5] FROM ghcr.io/alelevente/base_slim:latest                         0.0s
 => [internal] load build context       

### Checking if aggregator (including Mlfow server) is available

In [ ]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response: 200 - 


### 1.Testing unathorized access

In [19]:
res = requests.get("http://localhost:5100/experiments")
print(f"Experiments response: {res.status_code} - {res.text}")

Experiments response: 401 - You are not authenticated. Please see https://www.mlflow.org/docs/latest/auth/index.html#authenticating-to-mlflow on how to authenticate.


### 2. Testing authentication

In [32]:
# Adimin login
admin_username = "admin"
with open("../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
admin_password = admin_config["admin_password"]

res = requests.get("http://localhost:5100/#/experiments", auth=(admin_username, admin_password))
print(f"Experiments response with admin credentials: {res.status_code} - {res.text}")

Experiments response with admin credentials: 200 - <!doctype html><html lang="en"><head><meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1,shrink-to-fit=no"/><link rel="shortcut icon" href="./static-files/favicon.ico"/><meta name="theme-color" content="#000000"/><link rel="manifest" href="./static-files/manifest.json" crossorigin="use-credentials"/><title>MLflow</title><script defer="defer" src="static-files/static/js/main.e54ad566.js"></script><link href="static-files/static/css/main.b0f58ef9.css" rel="stylesheet"></head><body><noscript>You need to enable JavaScript to run this app.</noscript><div id="root" class="mlflow-ui-container"></div><div id="modal" class="mlflow-ui-container"></div></body></html>


In [64]:
def test_model_push():
    try:
        mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
        mlflow.sklearn.autolog()

        # Minimal dummy data for training a model
        x = np.array([1, 2, 3, 4, 5])
        y = np.array([2, 4, 5, 4, 5])

        model = LinearRegression()

        with mlflow.start_run() as run:
            model.fit(x.reshape(-1, 1), y)
            predictions = model.predict(x.reshape(-1, 1))
            mse = mean_squared_error(y, predictions)
            mae = mean_absolute_error(y, predictions)

            mlflow.log_metric("mse", mse)
            mlflow.log_metric("mae", mae)
    except Exception as e:
        print(f"Error during model push: {e}")

def test_model_pull():
    try:
        # Search the experiment and retrieve the runs
        experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        # Get the latest run
        latest_run = runs.iloc[0]
        # Load the model from the latest run
        model = mlflow.sklearn.load_model(f"runs:/{latest_run.run_id}/model")

        print(f"Loaded model from run ID: {latest_run.run_id}")
    except Exception as e:
        print(f"Failed to load model: {e}")

def remove_user_credentials():
    os.environ.pop("MLFLOW_TRACKING_USERNAME", None)
    os.environ.pop("MLFLOW_TRACKING_PASSWORD", None)
    print("Removed current user credentials from environment variables")
    try:
        print(os.environ['MLFLOW_TRACKING_USERNAME'])
    except KeyError:
        print("Username successfully removed from environment variables")

### 3. Test: Trainin and pushing a model
* a. as Aggregator user, the pushing should be successful.

In [62]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "aggregator"
print(f"Trying model push with username: {os.environ['MLFLOW_TRACKING_USERNAME']}")
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["aggregator_password"]

test_model_push()

remove_user_credentials()

Trying model push with username: aggregator


2026/06/25 16:16:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run useful-goose-234 at: http://localhost:5100/#/experiments/1/runs/5207bc66cf3040e49369d70c49a441d9
🧪 View experiment at: http://localhost:5100/#/experiments/1
Removed current user credentials from environment variables
Username successfully removed from environment variables


* b. as a simple Model User, should be unauthorized error (403)

In [68]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "model_user"
print(f"Trying model pull with username: {os.environ['MLFLOW_TRACKING_USERNAME']}")
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["model_user_password"]

test_model_push()

remove_user_credentials()

Trying model pull with username: model_user
Error during model push: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: 'Permission denied'
Removed current user credentials from environment variables
Username successfully removed from environment variables


* c. as an unauthenticated user, should be unauthenticated error (401)

In [70]:
test_model_push()

Error during model push: API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 401 != 200. Response body: 'You are not authenticated. Please see https://www.mlflow.org/docs/latest/auth/index.html#authenticating-to-mlflow on how to authenticate.'


### 4. Test: Pulling a trained and tracked model
* a. as a Model User, the model pulling should be successful 

In [71]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "model_user"
print(f"Trying model pull with username: {os.environ['MLFLOW_TRACKING_USERNAME']}")
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["model_user_password"]

test_model_pull()

remove_user_credentials()

Trying model pull with username: model_user
Loaded model from run ID: 5207bc66cf3040e49369d70c49a441d9
Removed current user credentials from environment variables
Username successfully removed from environment variables


* b. as Aggregator, should be successful as well

In [72]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "aggregator"
print(f"Trying model pull with username: {os.environ['MLFLOW_TRACKING_USERNAME']}")
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["aggregator_password"]

test_model_pull()

remove_user_credentials()

Trying model pull with username: aggregator
Loaded model from run ID: 5207bc66cf3040e49369d70c49a441d9
Removed current user credentials from environment variables
Username successfully removed from environment variables


c. as unauthenticated user it should be unauthenticated error (401)

In [73]:
test_model_pull()

Failed to load model: API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 401 != 200. Response body: 'You are not authenticated. Please see https://www.mlflow.org/docs/latest/auth/index.html#authenticating-to-mlflow on how to authenticate.'


### Stopping container, cleanup

In [74]:
!docker stop mlflow_auth_test

mlflow_auth_test
